In [ ]:
import os
import polars
from typing import Optional
from pathlib import Path
from datetime import date, timedelta
from git_author_stats import write_stats, iter_stats, read_stats

TODAY: date = date.today()
WEEK_START: date = TODAY - timedelta(days=TODAY.weekday())

# Because local commits, and commits which were in non-default branches, will
# not have been incorporated into our stats when this notebook was last run,
# we'll start looking for stats the *Monday* preceding 4 weeks ago today, in
# order to pick up stats from any commits which were subsequently merged.
# The date from which we'll retrieve stats, then, will either be the Monday
# preceding our last extracted stats, or the "tail" date, whichever is earlier.
# We will also strip previously retrieved stats for this time period from our
# previously extracted data, so that we don't double-count.
TAIL_START: date = WEEK_START - timedelta(
    days={{cookiecutter.stale_pull_request_open_days}}
)


def cut_off_tail() -> date:
    """
    Cut off the "tail" of our previously extracted stats, if needed, and return
    the date from which we should start looking for stats to extract.
    """
    since: date = date.today() - timedelta(
        days=INITIAL_HISTORY_NUMBER_OF_DAYS
    )
    # ...make it a Monday
    since -= timedelta(days=since.weekday())
    latest_path: Optional[Path] = None
    path: Path
    for path in filter(Path.is_file, Path("data").iterdir()):
        name: str
        extension: str
        name, extension = path.name.rpartition(".")[::2]
        if extension.lower() == "csv":
            file_date: date = date.fromisoformat(name)
            if file_date >= TAIL_START:
                path.unlink()
            elif (file_date >= since) or (latest_path is None):
                since = file_date
                latest_path = path
    if latest_path is not None:
        # Strip tail stats from our latest CSV file
        polars.LazyFrame(read_stats(latest_path)).filter(
            polars.col("before") < TAIL_START
        ).collect().write_csv(latest_path)
        # Get the first date not covered by our CSVs, now that we've
        # cut off their tail
        since = date.fromisoformat(
            polars.read_csv(latest_path).select(polars.max("before")).item()
        )
    return since


def update_stats() -> date:
    """
    Extract stats to a CSV in the data directory as needed to bring us
    up-to-date
    """
    since: date = cut_off_tail()
    # Lookup stats, starting the day after the last one covered by our last
    # previously retrieved records, and ending yesterday
    write_stats(
        iter_stats(
            "{{cookiecutter.repository_url}}",
            password=os.environ.get(
                "{{cookiecutter.token_environment_variable}}",
                ""
            ),
            since=since,
            # Don't include this week, since it isn't over yet
            before=WEEK_START,
            frequency="1w",
        ),
        f"data/{since.isoformat()}.csv",
    )
    return since


update_stats()

In [ ]:
from __future__ import annotations
import polars
import matplotlib.text
import matplotlib.ticker
import matplotlib.dates
import matplotlib.pyplot
import matplotlib.axes
import matplotlib.figure
from datetime import timedelta, date
from operator import itemgetter
from typing import Iterable, Sequence, Tuple, Union

LOCATOR: matplotlib.ticker.MultipleLocator = matplotlib.ticker.MultipleLocator(
    7.0
)
INITIAL_HISTORY_NUMBER_OF_DAYS: int = (
    {{cookiecutter.initial_history_number_of_days}}
)


def scan_stats(
    number_of_days: int = INITIAL_HISTORY_NUMBER_OF_DAYS / 2,
    include_author_names: tuple[str] | str = (),
    exclude_author_names: tuple[str] | str = (),
) -> polars.LazyFrame:
    if isinstance(include_author_names, str):
        include_author_names = (include_author_names,)
    if isinstance(exclude_author_names, str):
        exclude_author_names = (exclude_author_names,)
    stats: polars.LazyFrame = (
        polars.scan_csv("data/*.csv")
        # TODO: Modify the following filter to exclude commits/files
        # which you don't want to have contribute to your stats.
        .filter(
            # Config, JSON, requirement files, and the like are often
            # modified by project management tools
            polars.col("file").str.contains(
                r"(?i)\.(txt|ini|cfg|toml|yaml|yml|json)$"
            ).not_()
        )
        .select(
            (
                "url",
                (
                    polars.col("author_name")
                    .str.to_titlecase()
                    # TODO: Normalize author names here, if needed
                    # .replace(
                    #     "Anomalous Name Variation",
                    #     "Normalized Name",
                    # )
                ),
                polars.col("since").str.to_date(),
                polars.col("before").str.to_date(),
                "insertions",
                "deletions",
                "commit",
            )
        )
        .filter(
            polars.col("since") > date.today() - timedelta(days=number_of_days)
        )
    )
    if include_author_names:
        stats = stats.filter(
            polars.col("author_name").is_in(include_author_names)
        )
    if exclude_author_names:
        stats = stats.filter(
            polars.col("author_name").is_in(exclude_author_names).not_()
        )
    return stats


def scan_weekly_stats(
    number_of_days: int = INITIAL_HISTORY_NUMBER_OF_DAYS / 2,
    include_author_names: tuple[str] | str = (),
    exclude_author_names: tuple[str] | str = (),
) -> polars.LazyFrame:
    return (
        scan_stats(number_of_days, include_author_names, exclude_author_names)
        .group_by("url", "author_name", "since", "before", "commit")
        .sum()
        .select(("author_name", "since", "before", "insertions", "deletions"))
        .group_by("author_name", "since", "before")
        .sum()
        .sort(("since", "author_name"))
    )


def get_author_stats(
    number_of_days: int = INITIAL_HISTORY_NUMBER_OF_DAYS / 2,
    include_author_names: tuple[str] | str = (),
    exclude_author_names: tuple[str] | str = (),
    min_weeks: int = 2,
) -> polars.DataFrame:
    authors_data_frame: polars.DataFrame = (
        scan_weekly_stats(
            number_of_days, include_author_names, exclude_author_names
        )
        .select(
            (
                "author_name",
                "since",
                "before",
                "insertions",
                "deletions",
            )
        )
        .group_by("author_name")
        .agg(
            since=polars.col("since").min(),
            before=polars.col("before").max(),
            insertions=polars.sum("insertions"),
            deletions=polars.sum("deletions"),
        )
        .collect()
    )
    # Get the number of weeks each author_name has been active
    authors_data_frame = authors_data_frame.with_columns(
        weeks=(
            (polars.col("before") - polars.col("since")).dt.total_days() / 7
        ).cast(int)
    ).filter(polars.col("weeks") >= min_weeks)
    authors_data_frame = authors_data_frame.with_columns(
        weekly_insertions=(
            polars.col("insertions") / polars.col("weeks")
        ).cast(int),
        weekly_deletions=(polars.col("deletions") / polars.col("weeks")).cast(
            int
        ),
    )
    return authors_data_frame.sort("author_name", descending=True)



def get_weekly_stats(
    number_of_days: int = INITIAL_HISTORY_NUMBER_OF_DAYS / 2,
    include_author_names: tuple[str] | str = (),
    exclude_author_names: tuple[str] | str = (),
) -> polars.DataFrame:
    return (
        scan_weekly_stats(
            number_of_days, include_author_names, exclude_author_names
        )
        .group_by("since", "author_name")
        .sum()
        .sort("since")
        .select(
            (
                polars.col("since").alias("Week"),
                polars.col("author_name").alias("Author"),
                polars.col("insertions").alias("Insertions"),
                polars.col("deletions").alias("Deletions"),
            )
        )
    ).collect()


def get_sum_weekly_stats(
    number_of_days: int = INITIAL_HISTORY_NUMBER_OF_DAYS / 2,
    include_author_names: tuple[str] | str = (),
    exclude_author_names: tuple[str] | str = (),
) -> polars.DataFrame:
    return (
        scan_weekly_stats(
            number_of_days, include_author_names, exclude_author_names
        )
        .group_by("since")
        .sum()
        .sort("since")
        .select(
            (
                polars.col("since").alias("Week"),
                polars.col("insertions").alias("Insertions"),
                polars.col("deletions").alias("Deletions"),
            )
        )
    ).collect()


def get_mean_weekly_stats(
    number_of_days: int = INITIAL_HISTORY_NUMBER_OF_DAYS / 2,
    include_author_names: tuple[str] | str = (),
    exclude_author_names: tuple[str] | str = (),
) -> polars.DataFrame:
    return (
        scan_weekly_stats(
            number_of_days, include_author_names, exclude_author_names
        )
        .group_by("since")
        .mean()
        .sort("since")
        .select(
            (
                polars.col("since").alias("Week"),
                polars.col("insertions").alias("Insertions"),
                polars.col("deletions").alias("Deletions"),
            )
        )
        .collect()
    )


def get_sum_weekly_stats(
    number_of_days: int = INITIAL_HISTORY_NUMBER_OF_DAYS / 2,
    include_author_names: tuple[str] | str = (),
    exclude_author_names: tuple[str] | str = (),
) -> polars.DataFrame:
    """ """
    return (
        scan_weekly_stats(
            number_of_days, include_author_names, exclude_author_names
        )
        .group_by("since")
        .sum()
        .sort("since")
        .select(
            (
                polars.col("since").alias("Week"),
                polars.col("insertions").alias("Insertions"),
                polars.col("deletions").alias("Deletions"),
            )
        )
    ).collect()


def get_author_label(
    include_author_names: tuple[str] | str = (),
    exclude_author_names: tuple[str] | str = (),
) -> str:
    """
    Get the label to use based on included/excluded author names
    """
    label: list[str] = []
    author_names: tuple[str] = ()
    if exclude_author_names:
        if include_author_names:
            exclude_author_names_set: frozenset[str] = frozenset(
                exclude_author_names
            )
            author_names = tuple(
                author_name
                for author_name in include_author_names
                if author_name not in exclude_author_names_set
            )
        else:
            label.append("All Authors Except")
            author_names = exclude_author_names
    else:
        if include_author_names:
            author_names = include_author_names
        else:
            label.append("All Authors")
    if author_names:
        if len(author_names) == 1:
            label += (author_names[0],)
        else:
            label += "{} and {}".format(
                ", ".join(author_names[:-1]),
                author_names[-1],
            )
    return " ".join(label)


def plot_weekly_sum(
    number_of_days: int = INITIAL_HISTORY_NUMBER_OF_DAYS / 2,
    include_author_names: tuple[str] | str = (),
    exclude_author_names: tuple[str] | str = (),
) -> matplotlib.axes.Axes:
    author_label = get_author_label(include_author_names, exclude_author_names)
    axes: matplotlib.axes.Axes = (
        get_sum_weekly_stats(
            number_of_days, include_author_names, exclude_author_names
        )
        .to_pandas()
        .plot.area(
            title=(
                f"{author_label} - Weekly Total - "
                f"Past {number_of_days} Days"
            ),
            x="Week",
            y=["Insertions", "Deletions"],
            figsize=(32, 4),
            rot=90,
            xlabel="Week",
            ylabel="Insertions",
            x_compat=True,
        )
    )
    axes.xaxis.set_major_locator(LOCATOR)
    axes.xaxis.set_ticks_position("none")
    return axes


def plot_weekly_mean(
    number_of_days: int = INITIAL_HISTORY_NUMBER_OF_DAYS / 2,
    include_author_names: tuple[str] | str = (),
    exclude_author_names: tuple[str] | str = (),
) -> matplotlib.axes.Axes:
    author_label = get_author_label(include_author_names, exclude_author_names)
    axes: matplotlib.axes.Axes = (
        get_mean_weekly_stats(
            number_of_days, include_author_names, exclude_author_names
        )
        .to_pandas()
        .plot.area(
            title=(
                f"{author_label} - Weekly per/Author Average - "
                f"Past {number_of_days} Days"
            ),
            x="Week",
            y=["Insertions", "Deletions"],
            figsize=(32, 4),
            rot=90,
            xlabel="Week",
            ylabel="Insertions",
            x_compat=True,
        )
    )
    axes.xaxis.set_major_locator(LOCATOR)
    axes.xaxis.set_ticks_position("none")
    return axes


def plot_author_weekly_insertions(
    number_of_days: int = INITIAL_HISTORY_NUMBER_OF_DAYS / 2,
    include_author_names: tuple[str] | str = (),
    exclude_author_names: tuple[str] | str = (),
) -> Sequence[matplotlib.axes.Axes]:
    """
    Plot weekly insertions for all authors over the past `number_of_days`
    """
    author_label: str = get_author_label(
        include_author_names,
        exclude_author_names,
    )
    axes: matplotlib.axes.Axes
    authors_axes: Sequence[matplotlib.axes.Axes] = (
        get_weekly_stats(
            number_of_days, include_author_names, exclude_author_names
        )
        .to_pandas()
        .pivot(index="Week", columns="Author", values="Insertions")
        .plot.area(
            title=(
                f"Weekly Insertions - Past {number_of_days} Days - "
                f"{author_label}"
            ),
            figsize=(32, 16),
            rot=90,
            x_compat=True,
            subplots=True,
        )
    )
    # Get the minimum and maximum y-bounds for all axes, in order to display
    # all data at the same scale
    ybounds: Tuple[Tuple[float, float], ...] = tuple(
        map(matplotlib.axes.Axes.get_ybound, authors_axes)
    )
    ybound: Tuple[float, float] = (
        max(min(map(itemgetter(0), ybounds)), 0),
        max(map(itemgetter(1), ybounds)),
    )
    # Set the y-bounds for all axes to the same values
    for axes in authors_axes:
        axes.set_ybound(*ybound)
        axes.xaxis.set_ticks_position("none")
    authors_axes[-1].xaxis.set_major_locator(LOCATOR)
    return authors_axes


def plot_author_means(
    *numbers_of_days: int,
    include_author_names: tuple[str] | str = (),
    exclude_author_names: tuple[str] | str = (),
) -> Tuple[
    matplotlib.figure.Figure,
    Union[Sequence[matplotlib.axes.Axes], matplotlib.axes.Axes],
]:
    """
    Plot the weekly average insertions and deletions for all authors over
    the past `numbers_of_days` days (1 plot for each number of days provided)
    """
    author_label: str = get_author_label(
        include_author_names,
        exclude_author_names,
    )
    figure: matplotlib.figure.Figure
    axeses: Sequence[matplotlib.axes.Axes]
    number_of_columns: int = len(numbers_of_days)
    figure, axeses = matplotlib.pyplot.subplots(
        ncols=number_of_columns, figsize=(24, 8)
    )
    if isinstance(axeses, matplotlib.axes.Axes):
        axeses = (axeses,)
    index: int
    number_of_days: int
    for index, number_of_days in enumerate(numbers_of_days):
        get_author_stats(
            number_of_days=number_of_days,
            include_author_names=include_author_names,
            exclude_author_names=exclude_author_names,
        ).select(
            polars.col("author_name").alias("Author"),
            polars.col("weekly_insertions").alias("Weekly Insertions"),
            polars.col("weekly_deletions").alias("Weekly Deletions"),
        ).to_pandas().plot.barh(
            ax=axeses[index],
            x="Author",
            y=["Weekly Insertions", "Weekly Deletions"],
            stacked=True,
            title=(
                f"Past {number_of_days} Days: "
                f"Weekly Insertions and Deletions - {author_label}"
            ),
        )
    return figure, axeses


plot_author_means(
    # One plot will be produced for each number of days passed as a
    # positional argument
    INITIAL_HISTORY_NUMBER_OF_DAYS / 2,
    INITIAL_HISTORY_NUMBER_OF_DAYS,
    # If you want to exclude any authors, such as a service ID or bot,
    # you can exclude them through inclusion in a tuple passed to the
    # `exclude_author_names` argument:
    # exclude_author_names=('Service-id',),
)
plot_author_weekly_insertions(INITIAL_HISTORY_NUMBER_OF_DAYS)
plot_weekly_mean(number_of_days=INITIAL_HISTORY_NUMBER_OF_DAYS)
plot_weekly_sum(number_of_days=INITIAL_HISTORY_NUMBER_OF_DAYS)
matplotlib.pyplot.show()